[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/r-kowalczyk/graph-link-prediction/blob/feat/graph-sage-with-edges/notebooks/graphsage_runner.ipynb)

# GraphSAGE Link Prediction - Google Colab Runner

This notebook trains a GraphSAGE model for binary link prediction, exports a serving bundle, and optionally runs a quick smoke test of the API endpoints.

The serving bundle can be downloaded and served locally on a laptop without needing a GPU or retraining.

## What this notebook does

1. Installs dependencies and clones the repository.
2. Trains a GraphSAGE model using semantic text embeddings as node features.
3. Exports a self-contained serving bundle (model weights, graph tensors, node mappings).
4. Optionally runs the FastAPI service inside Colab for a quick smoke test.

## What you get

After training and exporting, the serving bundle directory contains:

- `model_state.pt`: GraphSAGE encoder and decoder weights
- `manifest.json`: node mappings, model configuration, and serving metadata
- `node_features.npy`: semantic node embeddings used for message passing
- `edge_index.npy`: graph adjacency used for message passing
- `resolver_cache.json`: on-disk cache for external gene name lookups

Download this folder and run `graph-lp serve --bundle-dir <path>` locally to start the prediction API.

## Prerequisites

**For the quickstart sanity run** you do not need any external data. The repository includes a tiny bundled CSV dataset.

**For the full dataset run** you need the data on Google Drive:

1. Download the data folder from [here](https://drive.google.com/drive/folders/1XbGVQiNid29Mxt1tjR7gitfxaVwJvc0t?usp=sharing).
2. Save the folder in the root of your Google Drive (see `configs/full.yaml` for expected paths).
3. Run the next cell to mount your Google Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Step 1: Install PyTorch and PyTorch Geometric

This cell installs PyTorch and PyTorch Geometric with matching CUDA wheels when a GPU runtime is available, or CPU wheels otherwise.

GraphSAGE training uses PyTorch Geometric for the SAGEConv layers and the RandomLinkSplit utility.

In [ ]:
import subprocess


def _has_nvidia_runtime() -> bool:
    try:
        subprocess.run(
            ["nvidia-smi"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=True,
        )
        return True
    except Exception:
        return False


if _has_nvidia_runtime():
    torch_index_url = "https://download.pytorch.org/whl/cu126"
    pyg_wheel_url = "https://data.pyg.org/whl/torch-2.8.0%2Bcu126.html"
    print("Detected NVIDIA runtime. Installing PyTorch 2.8.0 (CUDA 12.6 wheels).")
else:
    torch_index_url = "https://download.pytorch.org/whl/cpu"
    pyg_wheel_url = "https://data.pyg.org/whl/torch-2.8.0%2Bcpu.html"
    print("No NVIDIA runtime detected. Installing PyTorch 2.8.0 (CPU wheels).")

%pip install -q torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url {torch_index_url}
%pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f {pyg_wheel_url}
%pip install -q torch-geometric

import torch

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)

## Step 2: Clone and install the repository

The repository is cloned from the `feat/graph-sage-with-edges` branch (change `BRANCH` below if needed) and installed in editable mode so the `graph-lp` CLI is available.

In [ ]:
# Change BRANCH to "main" once the GraphSAGE feature is merged.
BRANCH = "feat/graph-sage-with-edges"

!git clone --branch {BRANCH} https://github.com/r-kowalczyk/graph-link-prediction.git
%cd graph-link-prediction

%pip install -q -e .

In [ ]:
# Confirm the CLI is available and configs are present.
!graph-lp --help
!ls -lah configs

## Step 3: Quickstart sanity run (CPU, bundled data)

This cell trains GraphSAGE on the tiny bundled dataset to confirm the installation is working.

- **Config**: `configs/quickstart.yaml`
- **Data**: bundled CSV files in the repository (12 nodes, 18 edges)
- **Device**: CPU
- **Output**: `artifacts_quickstart/<timestamp>/`

This should complete in under a minute. The AUC values on this tiny dataset are not meaningful; the purpose is to verify the pipeline runs end-to-end.

In [ ]:
!graph-lp train --config configs/quickstart.yaml --model graphsage --device cpu --seed 42
!graph-lp evaluate --config configs/quickstart.yaml --device cpu

## Step 4: Full GraphSAGE run (Drive data, GPU)

This cell trains GraphSAGE on the full dataset stored on Google Drive.

- **Config**: `configs/full.yaml`
- **Data**: expected under `/content/drive/MyDrive/graph-link-prediction-files/data/full`
- **Device**: `auto` (uses CUDA when a GPU runtime is available)
- **Embedding model**: set `EMBEDDING_MODEL` below to choose a different transformer for node text features. The default is `michiyasunaga/BioLinkBERT-large`.
- **Output**: written to Google Drive so the artefacts persist after Colab disconnects.

Skip this cell if you only want the quickstart sanity run.

In [ ]:
EMBEDDING_MODEL = "michiyasunaga/BioLinkBERT-large"
FULL_OUTPUT_DIRECTORY = "/content/drive/MyDrive/graph-link-prediction-files/artifacts_graphsage"

!graph-lp train \
    --config configs/full.yaml \
    --model graphsage \
    --device auto \
    --seed 42 \
    --output-dir {FULL_OUTPUT_DIRECTORY}

!graph-lp evaluate \
    --config configs/full.yaml \
    --device auto \
    --output-dir {FULL_OUTPUT_DIRECTORY}

## Step 5: Export the serving bundle

This cell exports a self-contained serving bundle from the most recent GraphSAGE run.

The bundle contains everything needed to serve predictions without retraining: model weights, graph tensors, node mappings, and metadata.

Set `RUN_DIRECTORY` to the specific run folder printed by the training output. If you leave it as `None`, the export command will automatically use the latest run under the output directory.

In [ ]:
import os

# Set this to the exact run directory path if you want to export a specific run.
# Leave as None to export from the latest run in the quickstart output directory.
RUN_DIRECTORY = None

# Determine which config and output directory to use based on whether a full
# run was completed (artefacts on Drive) or only the quickstart sanity run.
full_output = "/content/drive/MyDrive/graph-link-prediction-files/artifacts_graphsage"
if os.path.isdir(full_output):
    export_config = "configs/full.yaml"
    export_output_dir = full_output
else:
    export_config = "configs/quickstart.yaml"
    export_output_dir = None

export_command = f"graph-lp export --config {export_config}"
if RUN_DIRECTORY is not None:
    export_command += f" --run-dir {RUN_DIRECTORY}"
elif export_output_dir is not None:
    export_command += f" --output-dir {export_output_dir}"

!{export_command}

## Step 6: Download the serving bundle

The cell below zips the serving bundle and triggers a browser download.

If the bundle is on Google Drive (full run), you can also download it directly from the Drive web interface.

Once downloaded, serve it locally with:

```bash
uv run graph-lp serve --bundle-dir /path/to/serving_bundle --host 127.0.0.1 --port 8000
```

In [ ]:
import glob
import os

from google.colab import files

# Find the most recent serving bundle directory.
search_paths = [
    "/content/drive/MyDrive/graph-link-prediction-files/artifacts_graphsage/*/serving_bundle",
    "artifacts_quickstart/*/serving_bundle",
]

bundle_candidates = []
for search_path in search_paths:
    bundle_candidates.extend(sorted(glob.glob(search_path)))

if not bundle_candidates:
    print("No serving bundle found. Run the export step first.")
else:
    bundle_path = bundle_candidates[-1]
    zip_output_path = "/content/serving_bundle.zip"
    !cd "{os.path.dirname(bundle_path)}" && zip -r "{zip_output_path}" serving_bundle/
    print(f"Bundle zipped from: {bundle_path}")
    print(f"Zip file: {zip_output_path}")
    files.download(zip_output_path)

## Step 7 (optional): Smoke test the serving API inside Colab

This cell loads the exported bundle and runs a few test requests using the FastAPI test client, without starting a long-running server process.

This is useful to confirm the bundle works before downloading it.

In [ ]:
import glob
import json

from fastapi.testclient import TestClient

from graph_lp.server import create_app

# Find the most recent serving bundle.
search_paths = [
    "/content/drive/MyDrive/graph-link-prediction-files/artifacts_graphsage/*/serving_bundle",
    "artifacts_quickstart/*/serving_bundle",
]
bundle_candidates = []
for search_path in search_paths:
    bundle_candidates.extend(sorted(glob.glob(search_path)))

if not bundle_candidates:
    print("No serving bundle found. Run the export step first.")
else:
    bundle_path = bundle_candidates[-1]
    print(f"Loading bundle from: {bundle_path}")

    app = create_app(bundle_directory=bundle_path, device_name="cpu")
    client = TestClient(app)

    # Health check
    health_response = client.get("/healthz")
    print(f"\nHealth check: {health_response.status_code}")
    print(json.dumps(health_response.json(), indent=2))

    # Read the manifest to find two existing entity names for a test request.
    with open(f"{bundle_path}/manifest.json", "r") as f:
        manifest = json.load(f)
    existing_names = list(manifest["node_name_to_id"].keys())

    if len(existing_names) >= 2:
        name_a, name_b = existing_names[0], existing_names[1]
        print(f"\nScoring existing pair: '{name_a}' and '{name_b}'")
        pair_response = client.post(
            "/predict_link",
            json={"entity_a_name": name_a, "entity_b_name": name_b},
        )
        print(json.dumps(pair_response.json(), indent=2))

    if existing_names:
        print(f"\nTop-3 links for '{existing_names[0]}'")
        links_response = client.post(
            "/predict_links",
            json={"entity_name": existing_names[0], "top_k": 3},
        )
        print(json.dumps(links_response.json(), indent=2))

    # Test inductive mode with a new entity.
    print("\nInductive test: scoring a new entity against an existing one")
    inductive_response = client.post(
        "/predict_link",
        json={
            "entity_a_name": "Novel Test Entity",
            "entity_a_description": "A synthetic test entity for validating inductive serving",
            "entity_b_name": existing_names[0],
        },
    )
    print(json.dumps(inductive_response.json(), indent=2))

## Step 8: Cross-evaluate on ground_truth.csv test pairs

The GraphSAGE pipeline and the baseline pipeline use different evaluation protocols. The baseline scores pairs from `ground_truth.csv` (with predefined negatives), while GraphSAGE scores held-out edges from `RandomLinkSplit` (with sampled negatives). This makes their AUC numbers incomparable.

This cell runs the cross-evaluation script, which loads the trained GraphSAGE model and scores the exact same `ground_truth.csv` test pairs the baseline uses. It also computes a parameter-free semantic cosine similarity baseline, so you can see whether GraphSAGE message passing adds signal beyond raw text embedding similarity.

**What to look for in the output:**

- **GraphSAGE test ROC-AUC**: compare this directly against the baseline's test AUC (they are now on the same pairs).
- **Semantic cosine test ROC-AUC**: if GraphSAGE beats this, message passing is contributing structural information.
- **Delta (GS - cos)**: positive means the graph structure helps; near-zero or negative means it does not.

In [ ]:
import glob
import os

# Locate the most recent GraphSAGE run directory. The run directory contains
# the trained model state, metadata, and node features needed for scoring.
full_artefacts_path = "/content/drive/MyDrive/graph-link-prediction-files/artifacts_graphsage"
quickstart_artefacts_path = "artifacts_quickstart"

run_candidates = []
if os.path.isdir(full_artefacts_path):
    config_path = "configs/full.yaml"
    # Each timestamped subdirectory is a training run.
    for entry in sorted(os.listdir(full_artefacts_path)):
        candidate = os.path.join(full_artefacts_path, entry)
        if os.path.isfile(os.path.join(candidate, "graphsage_model_state.pt")):
            run_candidates.append(candidate)
elif os.path.isdir(quickstart_artefacts_path):
    config_path = "configs/quickstart.yaml"
    for entry in sorted(os.listdir(quickstart_artefacts_path)):
        candidate = os.path.join(quickstart_artefacts_path, entry)
        if os.path.isfile(os.path.join(candidate, "graphsage_model_state.pt")):
            run_candidates.append(candidate)

if not run_candidates:
    print("No GraphSAGE run directory found. Train a model first (steps 3 or 4).")
else:
    latest_run_directory = run_candidates[-1]
    print(f"Using run directory: {latest_run_directory}")
    print(f"Using config: {config_path}")
    !python scripts/cross_evaluate_on_ground_truth.py \
        --config {config_path} \
        --run-dir "{latest_run_directory}"